# Lab 4 · Ship It to the Depot

*If it only runs in your notebook, it isn't a product.*

**Your mission:** Turn Garabato's digit reader into a command-line tool that the night shift can run without Jupyter: one command to train a model, one to read a parcel, and errors a person (or a script at 3 a.m.) can act on.

**Reaching Putting It All Together (the end of the minimum path) is a complete lab. Everything after it is extra.**

**In this notebook**

1. Setup: run the first code cell
2. How this lab works: the markers ✏️ WRITE CODE and ▶️ JUST RUN, and the three parts CORE, OPTIONAL and ADVANCED
3. Previously on Garabato: a two-minute recap
4. What is a command-line tool? Who uses it, stdout and stderr, exit codes
5. Just enough theory
6. Two windows: digits.py and this notebook
7. Fast lane: the whole CORE on one table
8. The worked example W: the anatomy of a command
9. CORE: C1 → C2 → C3 → C4 → C5 → Putting It All Together → Run It in a Real Terminal
10. Check yourself: four exam-style questions (5 minutes)
11. After the CORE: AI wrote this main(), and Hand it to your neighbour
12. OPTIONAL (extra practice): ENG1, SCI1, BLD1
13. ADVANCED (for students who move faster): ENG2, SCI2, BLD2, the Challenge
14. AI-use declaration, Before you go, What's next

Data: MNIST (LeCun, Cortes, Burges), CC BY-SA 3.0, adapted. Garabato Parcels is fictional.

**Today you review Theory S7 (command-line tools: `argparse`, `main()`, stdout, stderr and exit codes), plus Theory S5 (`json` files) and Theory S13 (a file is a module: `import digits`, the `__name__` guard), by turning the digit reader into a tool the night shift runs from a terminal; the ADVANCED task BLD2 uses Theory S16 (FastAPI and Pydantic): a review if you've had S16, a preview if not.**

In [ ]:
# ▶️ JUST RUN · Run me first (Shift+Enter). You don't need to read this cell.
LAB = 4
KIT_URL = "https://github.com/xavitel/programming-lab/releases/download/lab4/lab4_kit.zip"   # filled in at build time if the kit is hosted; used only on Colab
import os, sys, platform
from pathlib import Path

def _find_kit(start):
    """Return the folder containing labkit/: here, up to 3 levels above, or 1 level below."""
    start = Path(start).resolve()
    for folder in [start] + list(start.parents)[:3]:
        if (folder / "labkit" / "__init__.py").is_file():
            return folder
    for child in sorted(p for p in start.iterdir() if p.is_dir()):
        if (child / "labkit" / "__init__.py").is_file():
            return child
    return None

print("Python", platform.python_version(), "|", sys.executable)
if sys.version_info < (3, 10):
    raise SystemExit("STOP: this kernel runs Python " + platform.python_version()
                     + ". The lab needs Python 3.10 or newer.\nFix: click the kernel name (top right) > "
                       "Select Another Kernel > Python Environments > pick 3.10+ (e.g. your course .venv). "
                       "START_HERE.md, Route A step 3, shows exactly where to click.")
if ".zip" in str(Path.cwd()).lower():
    raise SystemExit("STOP: the notebook was opened from inside the zip file.\nFix: close it, right-click "
                     "the zip > Extract All, then in VS Code use File > Open Folder on the extracted folder.")
IN_COLAB = "google.colab" in sys.modules
kit = _find_kit(Path.cwd())
if kit is None and IN_COLAB:
    import zipfile, urllib.request
    zips = sorted(Path("/content").glob("lab*_kit*.zip"))
    if not zips and KIT_URL:
        print("Colab: downloading the kit ...")
        urllib.request.urlretrieve(KIT_URL, "/content/kit.zip")
        zips = [Path("/content/kit.zip")]
    for z in zips:
        zipfile.ZipFile(z).extractall("/content")
    kit = _find_kit("/content")
if kit is None:
    raise SystemExit("STOP: I can't find the labkit folder.\nFix: open the WHOLE extracted kit folder in "
                     "VS Code (File > Open Folder), not only the notebook. On Colab, upload the kit zip "
                     "with the Files panel and run this cell again.")
os.chdir(kit)
if str(kit) not in sys.path:
    sys.path.insert(0, str(kit))
try:
    get_ipython().run_line_magic("load_ext", "autoreload")   # picks up edits in .py files
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass
import labkit
labkit.start(LAB)
from labkit import check, check_all, rescue, load_digits, show_digit

## How this lab works

**Two markers tell you what to do with every cell:**

| Marker | What it means |
|---|---|
| **✏️ WRITE CODE** | You write or change code here. Stubs start with `# ===== YOUR CODE HERE =====`: replace `pass` with your code. |
| **▶️ JUST RUN** | Provided code: run it (Shift+Enter) and read what it prints. You don't need to change it. |

Helper functions in `labkit` are already implemented: use them, don't modify them.

**Three parts:**

| Part | Who and when | Feedback |
|---|---|---|
| **CORE** | everyone, in order: C1 → C2 → C3 → C4 → C5 → Putting It All Together (the minimum path: reaching it is a complete lab), then Run It in a Real Terminal | `check()`, 4 hints, `rescue()` |
| **OPTIONAL** | extra practice when your CORE is done, any task, any order: ENG1, SCI1, BLD1 | `check()` and hints |
| **ADVANCED** | for students who move faster; open-ended, you can continue at home: ENG2, SCI2, BLD2, the Challenge | a sanity check, a score, or your own measurements |

**Three tools.**
- `check("C1", digits.build_parser)` answers PASS, NOT YET or NOT STARTED. It shows the tests it ran (TEST 1, TEST 2, …),
  with the inputs, the expected result and your result; on NOT YET, the first failing test and a likely cause.
- **Hints** sit under each task. Open them one at a time.
- `digits.build_parser = rescue("C1")` hands you a working version, so the rest of the notebook keeps working.

**Stuck for 7 minutes after Hint 3? Use `rescue()`**, carry on, and come back to the task later.

**The AI dial** (a recommendation, not a ban):
- CORE: AI as **tutor**. Ask *why*, not for the answer; Hint 4 gives you a prompt. Try on your own for 7–10 minutes first.
- OPTIONAL and ADVANCED tasks: AI as **reviewer**. Keep only what you can explain line by line.
- The Challenge (ADVANCED): AI as **pair programmer**. Verify with checks and tests, and declare it at the end.
- Always: you own every line you run, you verify it, and you **never paste personal data into an AI tool**.

Work alone or with a neighbour, your choice. Nothing here is graded or collected.

## Previously on Garabato

Two minutes of recap. You don't need any code from earlier labs.

**The story so far:** Lab 1 *I can code* → Lab 2 *I can write good Python* → Lab 3 *I can work with real data* → **Lab 4 (today)** *I can build software*

**Last time, in Lab 3:** you cleaned the scanner mess: CSV and JSON files, `None` for values you can't trust, cleaning rules in a fixed order, and proof that garbage in means garbage out.

**The ideas you keep using:**

- An image is a list of 784 numbers (0 = paper, 255 = ink).
- Distance = the sum of squared differences.
- Nearest neighbour = copy the label of the closest known image.
- Centroid = the average image of each digit.
- Accuracy = the fraction right on images the model never saw.
- Missed a lab? Everything you need is in `labkit.reference`.

## What is a command-line tool?

It is 3 a.m. at Garabato's Vallecas depot. The parcels of the night must be sorted before the vans leave at 6,
and the person on shift has never opened Jupyter. Ops sent one request: **"Ops doesn't do Jupyter. One command,
please."**

A **command-line tool** (a CLI) is a program you run by typing one line in a terminal: its name, then words
that say what to do. You have met some already: `pip install numpy`, `git commit -m "..."`, `uv sync`.
Today's will be `python digits.py predict models/centroids.json
samples/parcel_021.txt`.

**Two kinds of user.** A person types the command and reads the answer. And other programs run it with nobody
watching: a script, or a scheduler (cron on macOS and Linux, Task Scheduler on Windows) that starts the batch
at 3 a.m. That second user can't read sentences, so command-line tools keep a few strict habits:

| Habit | Why | Where in digits.py |
|---|---|---|
| words in: one subcommand per job, and `--help` | the person learns the tool from `--help` alone | `build_parser` (C1) |
| the model in a file | train once, predict many times, on any computer | `save_model` (C2; `load_model` is provided) |
| commands return text; one place prints it | the same code serves the terminal, the notebook and tests | `cmd_train`, `cmd_predict` (C3, C4) |
| results on **stdout**, problems on **stderr** | `... > result.txt` saves stdout: an error there would look like a result | `main` (C5) |
| an **exit code**: 0 worked, 1 an error the user can fix, 2 wrong words | a script reads the code, not the message | `main` (C5) |
| one error line, never a traceback | the night shift needs what to do ("use --force"), not Python's internals | `main` (C5) |

**Why not just a notebook?**

| | A notebook | A command-line tool |
|---|---|---|
| Who can run it | someone with Jupyter and the right kernel | anyone with Python, in a terminal |
| Can a scheduler run it at 3 a.m.? | not really | yes: one line |
| Can a test run it? | cell by cell, by hand | `main(["predict", ...])`, automatically |
| Can another program use its answer? | no | stdout and the exit code |

Today you build `digits.py` from the functions you already know (the average digit of Lab 1, files and JSON from
Lab 3), and run it two ways: from this notebook with `run_cli`, and from a real terminal.

## Just enough theory

Open a primer only if the idea feels new. Each one is a short example you can run in your head.

<details>
<summary><strong>argparse in five lines (Theory S7)</strong></summary>

A parser turns a list of words into `args`, a Namespace: one attribute per argument.

```python
import argparse
parser = argparse.ArgumentParser(prog="greet")
parser.add_argument("name")                            # positional: needed, by position
parser.add_argument("--times", type=int, default=1)    # an option with a value, converted to int
parser.add_argument("--shout", action="store_true")    # a flag: False unless typed
args = parser.parse_args(["Ana", "--times", "2"])      # Namespace(name='Ana', times=2, shout=False)
args.times + 1                                         # 3: an int, thanks to type=int
```

</details>

<details>
<summary><strong>Subcommands (Theory S7)</strong></summary>

One parser, several jobs, like `git commit` and `git push`. `dest="command"` stores which one was typed.

```python
subparsers = parser.add_subparsers(dest="command", required=True)
add = subparsers.add_parser("add", help="add two numbers")
add.add_argument("a", type=int)
add.add_argument("b", type=int)
args = parser.parse_args(["add", "2", "3"])      # args.command == "add", args.a == 2, args.b == 3
```

</details>

<details>
<summary><strong>main(argv=None) and the __name__ line (Theory S7)</strong></summary>

`main` takes the words as a list, so the notebook and tests can call it; `None` means "the words typed in the
terminal". The last two lines run it only when the file is run as a program, not when it is imported (the
`__name__` guard of Theory S13: a file is also a module).

```python
def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    ...
    return 0                      # the exit code

if __name__ == "__main__":        # True for `python digits.py ...`; False for `import digits`
    raise SystemExit(main())      # main's return value becomes the exit code
```

</details>

<details>
<summary><strong>stdout, stderr and exit codes (Theory S7)</strong></summary>

Two output streams: stdout for results, stderr for problems. The exit code is a number a script can test.

```python
import sys
print("Prediction: 7")                                       # stdout
print("error: models/x.json not found", file=sys.stderr)     # stderr
return 1                                                     # from main(): the exit code
```

</details>

<details>
<summary><strong>pathlib (Theory S7)</strong></summary>

A `Path` is a file name you can ask questions about; `/` joins its parts.

```python
from pathlib import Path
path = Path("models") / "centroids.json"
path.suffix, path.name, path.parent         # '.json', 'centroids.json', Path('models')
path.exists()                               # True or False
path.parent.mkdir(parents=True, exist_ok=True)   # create the folder (no error if it is there)
path.stat().st_size                         # the size in bytes
```

</details>

<details>
<summary><strong>json (Theory S5)</strong></summary>

JSON is text that any language can read. `dump`/`load` work on files, `dumps`/`loads` on strings.

```python
import json
with open("m.json", "w", encoding="utf-8") as f:
    json.dump({"format": "x", "version": 1}, f)        # dict -> file
json.loads('{"version": 1}')                           # text -> dict: {'version': 1}
json.loads("not json")          # raises json.JSONDecodeError, a kind of ValueError
```

</details>

<details>
<summary><strong>raise, and turning one error into another (Theory S2)</strong></summary>

`raise` stops a function and reports the problem to its caller; `try/except` catches a problem you expect.

```python
if path.exists() and not force:
    raise FileExistsError(f"{path} exists; use --force to overwrite")
try:
    model = json.loads(text)
except json.JSONDecodeError:
    raise ValueError(f"not a valid model file: {path}") from None   # a message the user can act on
```

</details>

<details>
<summary><strong>A stopwatch and number formatting (Theory S1)</strong></summary>

`time.perf_counter()` reads a stopwatch; the difference of two readings is the time in seconds. Inside an
f-string, what follows `:` says how to write the number.

```python
import time
start = time.perf_counter()
seconds = time.perf_counter() - start          # e.g. 1.2345...
f"{1234567:,}"                                 # '1,234,567'  (thousands separators)
f"{2.345:.1f}"                                 # '2.3'        (one decimal)
f"{48.6:.0f}"                                  # '49'         (a whole number, rounded)
```

</details>

<details>
<summary><strong>A dict of functions (Theory S3)</strong></summary>

A function is a value: you can store it in a dict, look it up, then call it.

```python
def hello(name):
    return f"hello {name}"

actions = {"hello": hello}
actions["hello"]("Ana")       # 'hello Ana': look the function up, then call it with ( )
```

</details>

## Two windows: digits.py and this notebook

In this lab your code lives in a **file**, `digits.py`, not in the notebook: a file is what the night shift can
run. It is also a **module** (Theory S13): the notebook imports it, the terminal runs it. The notebook keeps the
instructions, the checks, and a way to run your tool without leaving it.

1. **Open `digits.py`** (VS Code: click it in the Explorer on the left). Drag its tab to the right half of the
   window, or right-click the tab > **Split Right**, so you see the file and this notebook side by side.
   (Colab: open the Files panel, double-click `digits.py`; it opens in an editor next to the notebook.)
2. **Write** a function in `digits.py`. Each one you complete has the banner `# ===== YOUR CODE HERE =====` and the
   marker ✏️ WRITE CODE. Everything marked "Provided" is already implemented: use it, don't modify it.
3. **Save the file** (Ctrl+S, or Cmd+S on a Mac). A dot on the file's tab means "not saved": Python only sees
   what is saved.
4. **Run the check cell** of the task, here. It reads `digits.py` again, as you last saved it, so you never need to
   restart the kernel.

**The notebook route.** `run_cli(digits.main, "--help")` runs your tool exactly as `python digits.py --help` would
in a terminal: it shows what the tool prints, its error lines (marked `stderr |`) and its exit code. At the end of
the CORE you run the same commands in a real terminal.

In [ ]:
# ▶️ JUST RUN · open digits.py from the notebook (run it once; every check cell reads the file again)
from pathlib import Path
from labkit.cli import run_cli
from labkit.lab4_tools import check, check_all     # labkit's check and check_all, worded for digits.py
from labkit.lab4_tools import reload_digits, call_command, try_model_file, show_model_file, read_samples, terminal
digits = reload_digits()                  # imports digits.py as it is saved now
print("Loaded", Path(digits.__file__).name, "with", len([n for n in dir(digits) if n.startswith("cmd_")]),
      "command functions so far")

In [ ]:
# ▶️ JUST RUN · your tool, from minute 1: the same as typing `python digits.py --help` in a terminal
run_cli(digits.main, "--help");

## Fast lane

Some people like to read a spec and build; others like guided steps. Both routes reach the same place.

| Task | Signature | Spec |
|---|---|---|
| C1 (minimum path) | `build_parser() -> argparse.ArgumentParser` | prog "digits"; subcommands (dest="command", required=True): inspect (given), train DATA --out MODEL [--limit N] [--force], predict MODEL IMAGE [--show] |
| C2 (minimum path) | `save_model(model: dict, path: Path, force: bool = False) -> None` | FileExistsError("... exists; use --force to overwrite") unless force; create the folder (exist_ok=True); UTF-8 JSON with json.dump. (load_model is provided.) |
| C3 (minimum path) | `cmd_train(args) -> str` | load_digits(args.data, limit=args.limit) -> new_model(fit_centroids(...)) -> save_model(..., force=args.force); return "Trained on N images in S s: saved OUT (K KB)" |
| C4 (minimum path) | `cmd_predict(args) -> str` | load_model; read by suffix (.csv read_image_csv, .txt read_image_grid, else ValueError("unsupported image type: ...")); return "Prediction: 7" (with --show: picture(image), a newline, then that line) |
| C5 (minimum path) | `main(argv: list[str] \| None = None) -> int` | parse; dispatch dict; print the result and return 0; FileNotFoundError, FileExistsError, ValueError -> print(f"error: {err}", file=sys.stderr), return 1 |

(minimum path) marks the tasks you need for Putting It All Together: C1 → C2 → C3 → C4 → C5. Each function has a stub in `digits.py`, under the banner `# ===== YOUR CODE HERE =====`: write your code there, save the file, then run the cell below at any time to see where you are. Each line tells you the next step. After Putting It All Together, the last CORE step is Run It in a Real Terminal.

In [ ]:
# ▶️ JUST RUN · where am I? (it reads digits.py again first)
digits = reload_digits()
check_all();   # the ; only stops Jupyter from printing the same result again as a dict

### W · Worked example: The anatomy of a command · ▶️ JUST RUN

The worked example: the inspect subcommand is already written in digits.py. Follow one command line from the words typed to the exit code.

**You'll practise:** the shape of every command: args in, a string out (Theory S7)

**Signature**

```python
def cmd_inspect(args: argparse.Namespace) -> str      # in digits.py, already written
```

**Toy example**

```python
# python digits.py inspect data/mnist_test_2k.csv.gz --index 3
cmd_inspect(args)    # returns the text: the file, its labels, and a picture of image 3
```

Open `digits.py` and find `cmd_inspect` (and the inspect part of `build_parser`): read them, then run the cells below.

```text
python digits.py inspect data/mnist_test_2k.csv.gz --index 3      <- the words typed in a terminal
   |
   v  parser = build_parser(); args = parser.parse_args(words)       C1 (inspect is already there)
Namespace(command='inspect', data='data/mnist_test_2k.csv.gz', index=3)
   |
   v  main() looks the command up: {"inspect": cmd_inspect, ...}     C5
cmd_inspect(args)  -> returns a string                              W (C3 and C4 have the same shape)
   |
   v  main() prints the string and returns 0                         C5
exit code 0                                                         (1: an error the user can fix; 2: wrong words)
```

In [ ]:
# ▶️ JUST RUN · the anatomy of a command, one step at a time (read cmd_inspect in digits.py while it runs)
parser = digits.build_parser()                                             # 1. the parser knows the words
args = parser.parse_args(["inspect", "data/mnist_test_2k.csv.gz", "--index", "3"])   # 2. words -> args
print(args)                                                                # a Namespace: args.data, args.index
text = digits.cmd_inspect(args)                                            # 3. the command RETURNS a string
print(text)                                                                # 4. main() prints it (C5)

In [ ]:
# ▶️ JUST RUN · check the worked example (it passes: it shows you what a check looks like)
digits = reload_digits()
check("W", digits.cmd_inspect);

**Notice three habits in `cmd_inspect`, which every command of today shares:** it gets everything it needs
from `args`; it **returns** its text (it never prints); and when something is wrong (an index outside the
file) it **raises** an error with a message the user can act on, and never calls `sys.exit()`. `main()`
(C5) decides what to print and which exit code to return.

**Try it (1 minute).** In the cell above, change `"3"` to `"1999"`, run it; then `"2000"`. Read the last
line of the error: it says what the allowed indices are.

## CORE: the tasks everyone does

**C1 → C2 → C3 → C4 → C5 → Putting It All Together → Run It in a Real Terminal.** The first six are the **minimum path**: reaching Putting It All Together is a complete lab. Run It in a Real Terminal completes the CORE.

For each task: read the card; write the function in `digits.py` (its stub sits under the banner `# ===== YOUR CODE HERE =====`: replace `pass`, or complete the lines that are there); **save the file**; then run the check cell under the card. The check reads `digits.py` again and shows the tests it ran: what each one checks, its inputs, the expected result and your result. Then a view cell shows what your function does with real data.

**The data today:** the tool learns from 10,000 examples of handwritten digits; the quick views use the first 1,000 to stay fast.

### C1 · Words in, args out · ✏️ WRITE CODE

The night shift types words; your parser turns them into args. inspect is already in build_parser: add train and predict.

**You'll practise:** argparse: subcommands, positional arguments, options with a type, flags (Theory S7) · **Typical time:** 6–9 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def build_parser() -> argparse.ArgumentParser      # in digits.py
```

**Toy example**

```python
# train DATA --out MODEL [--limit N] [--force]
digits train data/mnist_train_10k.csv.gz --out models/centroids.json --limit 500
#   -> Namespace(command='train', data='data/mnist_train_10k.csv.gz', out='models/centroids.json',
#                limit=500, force=False)
# predict MODEL IMAGE [--show]
digits predict models/centroids.json samples/parcel_021.txt --show
#   -> Namespace(command='predict', model='models/centroids.json', image='samples/parcel_021.txt', show=True)
```

**Three kinds of argument, three jobs:**

| Kind | How you add it | Typed as | Use it for |
|---|---|---|---|
| positional | `add_argument("data")` | `train data/x.csv.gz` | what every run needs, in a fixed order: the data, the model, the image |
| option with a value | `add_argument("--out", required=True)`, `add_argument("--limit", type=int, default=None)` | `--out models/c.json`, `--limit 500` | named settings, in any order; `type=int` converts the text and rejects `--limit ten` |
| flag | `add_argument("--force", action="store_true")` | `--force` | yes or no: `False` unless typed |

**Why argparse, and not `sys.argv` by hand?** One line per argument gives you `--help` for free, the type
checks, and a clear usage error (exit code 2) when the words are wrong. By hand, each of those is your code to
write and test.

**Where:** `digits.py`, the function `build_parser` (look for the banner `C1`). Write it there, **save the file** (Ctrl+S, or Cmd+S on a Mac), then run the check cell below.

**Stuck for 7 minutes after Hint 3?** Run `digits.build_parser = rescue("C1")` in a new cell: your digits.py then uses the reference build_parser (until you edit yours). Carry on, and come back to C1 later.

In [ ]:
# ▶️ JUST RUN · check your work (this cell reads digits.py again, as you last saved it)
digits = reload_digits()
check("C1", digits.build_parser);

In [ ]:
# ▶️ JUST RUN · see it: the help your parser writes for the night shift (before C1: inspect only)
run_cli(digits.main, "--help");
run_cli(digits.main, "train --help");

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Copy the pattern of `inspect`, a few lines above in `build_parser`: one `add_parser` for the subcommand, then one `add_argument` per word. Positional arguments have no dashes; options start with `--` (Theory S7).

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `train = subparsers.add_parser("train", help="learn a model and save it")`.
2. `train.add_argument("data")`, `train.add_argument("--out", required=True)`, `train.add_argument("--limit", type=int, default=None)`, `train.add_argument("--force", action="store_true")`.
3. `predict = subparsers.add_parser("predict", help="read one image")`, then `predict.add_argument("model")`, `predict.add_argument("image")`, `predict.add_argument("--show", action="store_true")`.
4. Keep `return parser` as the last line, save the file, run the check.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    train = subparsers.add_parser("train", help="learn a model and save it")
    train.add_argument(___)
    train.add_argument("--out", ___=True)
    train.add_argument("--limit", type=___, default=None)
    train.add_argument("--force", action=___)
    predict = subparsers.add_parser(___, help="read one image")
    predict.add_argument("model")
    predict.add_argument(___)
    predict.add_argument("--show", action="store_true")

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### C2 · A model is a file · ✏️ WRITE CODE

Train once, predict many times: the model must live in a file that any computer can read, and must never be overwritten by accident.

**You'll practise:** json, pathlib and raise: saving a file safely, and refusing to overwrite it (Theory S5–S7) · **Typical time:** 4–6 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def save_model(model: dict, path: Path, force: bool = False) -> None      # in digits.py
```

**Toy example**

```python
save_model(model, Path("models/centroids.json"))      # creates models/, writes the JSON file
save_model(model, Path("models/centroids.json"))      # FileExistsError: models/centroids.json exists; use --force to overwrite
save_model(model, Path("models/centroids.json"), force=True)   # overwrites
load_model(Path("models/centroids.json"))             # provided: the same dict back
```

**What's in a model file** (the provided `new_model()` builds this dict; the keys are text, because JSON keys
always are):

```json
{"format": "garabato-centroids", "version": 1, "created": "2026-11-18T15:20:05", "n_train": 10000,
 "image_size": 28, "centroids": {"0": [0.0, 0.0, 0.13, ...784 numbers], "1": [...], ..., "9": [...]}}
```

**Why JSON?**

| Format | A person can read it | Any language can read it | Safe to open a file from someone else |
|---|---|---|---|
| JSON | yes, in any editor | yes | yes: it holds only data |
| pickle (Python's own) | no | Python only | no: loading a pickle can run code |
| CSV | yes | yes | yes, but it's one table: no place for the format, the version, or ten lists |

**Its partner, `load_model`, is provided** (at the top of `digits.py`): read it. It checks `"format"` first,
because a JSON file from another tool would load without a complaint and fail later, deep in `predict`, with a
`KeyError` nobody understands; and it turns `json.JSONDecodeError` into one clear `ValueError`.

**Why refuse to overwrite?** Training can take minutes and the old model may be the one in use: `--force` says
"I mean it".

**Where:** `digits.py`, the function `save_model` (look for the banner `C2`). Write it there, **save the file** (Ctrl+S, or Cmd+S on a Mac), then run the check cell below.

**Stuck for 7 minutes after Hint 3?** Run `digits.save_model = rescue("C2")` in a new cell: your digits.py then uses the reference save_model (until you edit yours). Carry on, and come back to C2 later.

In [ ]:
# ▶️ JUST RUN · check your work (this cell reads digits.py again, as you last saved it)
digits = reload_digits()
check("C2", digits.save_model);

In [ ]:
# ▶️ JUST RUN · see it: a real model saved by YOUR save_model, the file, and the provided load_model reading it back
try_model_file(digits)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Three steps, in this order: check (refuse to overwrite), create the folder, write the JSON. A few lines with pathlib and json (Theory S5, Theory S7).

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `path = Path(path)`; if `path.exists() and not force`, raise `FileExistsError(f"{path} exists; use --force to overwrite")` (before opening: opening with `"w"` empties the file).
2. `path.parent.mkdir(parents=True, exist_ok=True)`: `exist_ok=True` means "no error if the folder is already there".
3. `with open(path, "w", encoding="utf-8") as f:` and `json.dump(model, f)`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    def save_model(model, path, force=False):
        path = Path(path)
        if path.exists() and not ___:
            raise FileExistsError(f"{path} exists; use --force to overwrite")
        path.parent.mkdir(parents=True, exist_ok=___)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(___, f)

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### C3 · Train from the command line · ✏️ WRITE CODE

The first command the night shift needs: learn the average image of each digit from a data file, and save it as a model file.

**You'll practise:** composing functions you are given, and returning a message instead of printing it (Theory S7) · **Typical time:** 5–7 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def cmd_train(args: argparse.Namespace) -> str      # in digits.py
```

**Toy example**

```python
# python digits.py train data/mnist_train_10k.csv.gz --out models/centroids.json --limit 1000
cmd_train(args)
# returns one line: 'Trained on 1,000 images in S.S s: saved models/centroids.json (K KB)'
```

**Four provided pieces, one new line of thought.** `load_digits(path, limit=...)` (labkit) reads the images,
`fit_centroids(images, labels)` (Lab 1) computes the average image of each digit, `new_model(...)` (in
`digits.py`) wraps them in the model dict, and your `save_model` (C2) writes it.

**Commands return; main() prints.** Why not `print` inside `cmd_train`?

| | `print(...)` inside the command | `return` the message |
|---|---|---|
| main() decides where it goes (stdout, a log, nowhere) | no | yes |
| a test can check it: `assert cmd_train(args).startswith("Trained on")` | no | yes |
| the notebook and other code can reuse it | no | yes |

**Where:** `digits.py`, the function `cmd_train` (look for the banner `C3`). Write it there, **save the file** (Ctrl+S, or Cmd+S on a Mac), then run the check cell below.

**Stuck for 7 minutes after Hint 3?** Run `digits.cmd_train = rescue("C3")` in a new cell: your digits.py then uses the reference cmd_train (until you edit yours). Carry on, and come back to C3 later.

In [ ]:
# ▶️ JUST RUN · check your work (this cell reads digits.py again, as you last saved it)
digits = reload_digits()
check("C3", digits.cmd_train);

In [ ]:
# ▶️ JUST RUN · see it: YOUR cmd_train on 1,000 training images, then the file it wrote
call_command(digits.cmd_train, "train data/mnist_train_10k.csv.gz --out models/centroids.json --limit 1000 --force");
show_model_file("models/centroids.json")

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Four calls in a row, each using what the one before returned. Start a stopwatch first (`time.perf_counter()`), and build the message at the end with one f-string.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `start = time.perf_counter()`, then `images, labels = load_digits(args.data, limit=args.limit)`.
2. `model = new_model(fit_centroids(images, labels), n_train=len(images))`.
3. `save_model(model, Path(args.out), force=args.force)`.
4. `seconds = time.perf_counter() - start`; `size_kb = Path(args.out).stat().st_size / 1024`; return `f"Trained on {len(images):,} images in {seconds:.1f} s: saved {args.out} ({size_kb:.0f} KB)"`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    start = time.perf_counter()
    images, labels = load_digits(args.data, limit=___)
    model = new_model(fit_centroids(___, ___), n_train=len(images))
    save_model(model, Path(args.out), force=___)
    seconds = time.perf_counter() - ___
    size_kb = Path(args.out).stat().st_size / 1024
    return f"Trained on {len(images):,} images in {seconds:.1f} s: saved {args.out} ({___:.0f} KB)"

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### C4 · Read one parcel · ✏️ WRITE CODE

The second command: read the digit in one parcel image. Images come as .csv or .txt files; anything else must be refused, politely.

**You'll practise:** choosing what to do from a file's suffix, with pathlib (Theory S7) · **Typical time:** 5–8 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def cmd_predict(args: argparse.Namespace) -> str      # in digits.py
```

**Toy example**

```python
# python digits.py predict models/centroids.json samples/parcel_021.txt
cmd_predict(args)     # 'Prediction: N'
# python digits.py predict models/centroids.json samples/parcel_021.txt --show
cmd_predict(args)     # picture(image) + "\n" + 'Prediction: N'
# python digits.py predict models/centroids.json photo.png
cmd_predict(args)     # ValueError: unsupported image type: .png (use .csv or .txt)
```

**Two image formats, one rule: the suffix decides.** `Path(args.image).suffix` gives the suffix with its dot.

| Suffix | What's inside | Reader (provided) |
|---|---|---|
| `.csv` | one line of 784 numbers, separated by commas | `read_image_csv(path)` |
| `.txt` | 28 lines of 28 numbers, separated by spaces (what `tools/draw_digit.html` saves) | `read_image_grid(path)` |
| anything else | unknown | refuse: `raise ValueError(f"unsupported image type: {suffix} (use .csv or .txt)")` |

**Why refuse instead of guessing?** A guess that goes wrong gives a wrong postcode on a parcel, silently. A refusal
gives the night shift one line that says what to do. (`--show` draws the image with the provided `picture()`, so
a person can look at what the machine read.)

**Where:** `digits.py`, the function `cmd_predict` (look for the banner `C4`). Write it there, **save the file** (Ctrl+S, or Cmd+S on a Mac), then run the check cell below.

**Stuck for 7 minutes after Hint 3?** Run `digits.cmd_predict = rescue("C4")` in a new cell: your digits.py then uses the reference cmd_predict (until you edit yours). Carry on, and come back to C4 later.

In [ ]:
# ▶️ JUST RUN · check your work (this cell reads digits.py again, as you last saved it)
digits = reload_digits()
check("C4", digits.cmd_predict);

In [ ]:
# ▶️ JUST RUN · see it: YOUR cmd_predict reads one parcel (a .csv one) and draws it (it uses the model the C3 cell saved)
call_command(digits.cmd_predict, "predict models/centroids.json samples/parcel_013.csv --show");

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Three steps: load the model (the provided `load_model`), read the image with the reader its suffix asks for, predict with the provided `predict_centroid`. Return the text; with `args.show`, the picture goes first.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `model = load_model(Path(args.model))` (provided).
2. `suffix = Path(args.image).suffix`; if `suffix == ".csv"`: `image = read_image_csv(args.image)`; elif `suffix == ".txt"`: `image = read_image_grid(args.image)`; else raise `ValueError(f"unsupported image type: {suffix} (use .csv or .txt)")`.
3. `label = predict_centroid(image, centroids_of(model))`.
4. If `args.show`: return `picture(image) + "\n" + f"Prediction: {label}"`; otherwise return `f"Prediction: {label}"`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    model = load_model(Path(args.model))
    suffix = Path(args.image).___
    if suffix == ".csv":
        image = read_image_csv(args.image)
    elif suffix == ___:
        image = read_image_grid(args.image)
    else:
        raise ValueError(f"unsupported image type: {suffix} (use .csv or .txt)")
    label = predict_centroid(image, centroids_of(___))
    if args.show:
        return picture(image) + "\n" + f"Prediction: {label}"
    return ___

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### C5 · One entry point, honest exit codes · ✏️ WRITE CODE

One entry point for every command: main() parses the words, runs the right command, prints its result, and returns an exit code that a script can trust.

**You'll practise:** the main pattern: a dispatch dict, stdout vs stderr, exit codes (Theory S7) · **Typical time:** 7–10 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def main(argv: list[str] | None = None) -> int      # in digits.py; the first two lines are there
```

**Toy example**

```python
main(["predict", "models/centroids.json", "samples/parcel_021.txt"])   # prints 'Prediction: N', returns 0
main(["predict", "models/nope.json", "samples/parcel_021.txt"])        # stderr: error: ... nope.json ..., returns 1
main(["train"])                                                       # argparse: usage on stderr, exit code 2
```

**Who reads what.** A person reads the words; a script that runs `digits.py` at 3 a.m. reads only the exit code
and stdout (often saved to a file: `python digits.py predict ... > result.txt`).

| What happened | stdout | stderr | exit code |
|---|---|---|---|
| it worked | the result | nothing | **0** |
| a problem the user can fix: a missing file, `--force` needed, not a model, an image type it can't read | nothing | ONE line: `error: ...` | **1** |
| the command line itself is wrong | nothing | argparse's usage message | **2** (argparse does it) |

**Why a dispatch dict?** Functions are values, so a dict can map each subcommand to its function:
`commands = {"inspect": cmd_inspect, "train": cmd_train, "predict": cmd_predict}`, then
`commands[args.command](args)`. A new subcommand is one more entry, not one more `elif`.

**Why never `sys.exit()` inside a command?** It stops the whole program, so `main([...])` in a test or in this
notebook never returns. Commands raise; main() catches the errors it expects (`FileNotFoundError`,
`FileExistsError`, `ValueError`; `json.JSONDecodeError` is a kind of `ValueError`) and returns the code.

**Where:** `digits.py`, the function `main` (look for the banner `C5`). Write it there, **save the file** (Ctrl+S, or Cmd+S on a Mac), then run the check cell below.

**Stuck for 7 minutes after Hint 3?** Run `digits.main = rescue("C5")` in a new cell: your digits.py then uses the reference main (until you edit yours). Carry on, and come back to C5 later. (A rescued main runs the reference commands.)

In [ ]:
# ▶️ JUST RUN · check your work (this cell reads digits.py again, as you last saved it)
digits = reload_digits()
check("C5", digits.main);

In [ ]:
# ▶️ JUST RUN · see it: YOUR main(), as a terminal would run it: a result, then an error
run_cli(digits.main, "predict models/centroids.json samples/parcel_013.csv");
run_cli(digits.main, "predict models/nope.json samples/parcel_013.csv");

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Three TODOs under the two given lines: a dict of functions, one call inside try/except, and the right `return` in each branch. Errors go to stderr: `print(..., file=sys.stderr)` (Theory S7).

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `commands = {"inspect": cmd_inspect, "train": cmd_train, "predict": cmd_predict}`.
2. `try:` `output = commands[args.command](args)`.
3. `except (FileNotFoundError, FileExistsError, ValueError) as err:` `print(f"error: {err}", file=sys.stderr)` and `return 1`.
4. After the try/except: `print(output)` and `return 0`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    commands = {"inspect": cmd_inspect, "train": ___, "predict": ___}
    try:
        output = commands[___](args)
    except (FileNotFoundError, FileExistsError, ValueError) as err:
        print(f"error: {err}", file=___)
        return ___
    print(output)
    return ___

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### Putting It All Together · ▶️ JUST RUN

This is where your five functions work together, exactly as the night shift will use them: three commands, each
through `main()`. We'll train on 10,000 examples of handwritten digits, then read a parcel from the `samples/`
folder.

1. **train**: `main` (C5) parses the words with `build_parser` (C1) and runs `cmd_train` (C3), which saves the
   model with `save_model` (C2);
2. **predict --show**: `main` runs `cmd_predict` (C4), which loads the model with the provided `load_model`, reads
   the parcel and draws it;
3. **a mistake**: a model file that isn't there. What does the night shift see, and what does a script see?

`run_cli` runs each one as `python digits.py ...` would. What will your tool say about the parcel? Run them and find
out.

In [ ]:
# ▶️ JUST RUN · Putting It All Together, 1 of 3: train on the 10,000 training images (a few seconds)
run_cli(digits.main, "train data/mnist_train_10k.csv.gz --out models/centroids.json --force");

In [ ]:
# ▶️ JUST RUN · Putting It All Together, 2 of 3: read one parcel and draw it
run_cli(digits.main, "predict models/centroids.json samples/parcel_021.txt --show");

In [ ]:
# ▶️ JUST RUN · Putting It All Together, 3 of 3: a mistake, a model file that isn't there
run_cli(digits.main, "predict models/nope.json samples/parcel_021.txt");

**Reaching Putting It All Together is a complete lab.** The same code now has two faces: a function you can call
(`main([...])`, for the notebook and for tests) and a command anyone can type. The error was one line on stderr
with exit code 1, so a script knows something went wrong, and a person knows what.

Next, the last CORE step: run it in a real terminal. Curious first? The cell below reads every parcel in
`samples/` through your `main()`. Pick two answers you would question and look at them with `--show`: would you
have read them the same?

In [ ]:
# ▶️ JUST RUN (optional) · every parcel in samples/ through YOUR main(), one line each
read_samples(digits.main)

### Run It in a Real Terminal · ▶️ JUST RUN

**The last CORE step, about 5 minutes.** Same code, a new face: the night shift doesn't open notebooks.

1. **Open a terminal in the kit folder.** VS Code: **Terminal > New Terminal**; it opens in the folder you opened
   (the one with `digits.py` and `labkit`). Not sure? `dir` (Windows) or `ls` (macOS) should list `digits.py`;
   if not, `cd` into the kit folder first.
2. **Type these three commands** (the first word depends on your system):

| System | Help | Train | Read a parcel |
|---|---|---|---|
| Windows | `py digits.py --help` | `py digits.py train data/mnist_train_10k.csv.gz --out models/centroids.json --force` | `py digits.py predict models/centroids.json samples/parcel_021.txt --show` |
| macOS, Linux | `python3 digits.py --help` | `python3 digits.py train data/mnist_train_10k.csv.gz --out models/centroids.json --force` | `python3 digits.py predict models/centroids.json samples/parcel_021.txt --show` |
| with uv | `uv run python digits.py --help` | `uv run python digits.py train ...` (as above) | `uv run python digits.py predict ...` (as above) |
| Colab | `!python digits.py --help` in a code cell | `!python digits.py train ...` | `!python digits.py predict ...` |

3. **The exit code of the last command:** `$LASTEXITCODE` (Windows PowerShell) or `echo $?` (macOS, Linux). Try
   `predict models/nope.json samples/parcel_021.txt`, then the exit code: 1.

**Used `rescue()`?** A rescue lives only in this notebook: the terminal runs `digits.py` as saved. Finish the
rescued function in `digits.py` first (or copy it from `labkit/reference/lab4.py`), and save.

**`python` not found?** On Windows, type `py` instead. **`No module named labkit`?** The terminal is in another
folder: `cd` into the kit folder. **A different Python?** The check below prints the path of this notebook's
Python; you can type that path instead of `python`.

Then run the check: it runs `--help`, `inspect`, then `train` (on 50 images) and `predict` with this notebook's
Python, the way your terminal does.

In [ ]:
# ▶️ JUST RUN · check: digits.py runs as a program (--help, inspect, train, predict)
check("CP3");

**CORE complete.** Your tool reads parcels from a terminal, with a model saved as a file, a help text, and
errors a person and a script can act on. Next, five minutes on paper (Check yourself); then two short
activities: AI wrote this main(), and Hand it to your neighbour. Then any OPTIONAL task, or ADVANCED if you move
faster.

## Check yourself (5 minutes, exam-style)

Exam-style questions on today's ideas. Answer on paper or in your head first, then open the answer: nothing to run, nothing is checked.

**1. What does this print? And what would `args.limit + 1` do without `type=int`?**

```python
import argparse
parser = argparse.ArgumentParser(prog="digits")
parser.add_argument("image")
parser.add_argument("--limit", type=int, default=None)
parser.add_argument("--show", action="store_true")
args = parser.parse_args(["photo.txt", "--limit", "50"])
print(args.limit + 1, args.show, args.image)
```

<details>
<summary><strong>Answer</strong></summary>

`51 False photo.txt`. Every word typed is text: `type=int` turns `"50"` into `50`, and a flag that isn't typed is `False`. Without `type=int`, `args.limit` is the text `"50"`, and `"50" + 1` raises a `TypeError`.

</details>

**2. What does this print?**

```python
import json
model = {"centroids": {7: [0.5, 1.0]}}
text = json.dumps(model)
back = json.loads(text)
print(text)
print(7 in back["centroids"], "7" in back["centroids"])
```

<details>
<summary><strong>Answer</strong></summary>

`{"centroids": {"7": [0.5, 1.0]}}`, then `False True`. JSON keys are always text, so the int `7` comes back as `"7"`. That's why the model file stores `"0"` to `"9"`, and `centroids_of` turns them back into ints before predicting.

</details>

**3. What is wrong here? This is the end of a `digits.py`; the notebook (or a test) runs `import digits`.**

```python
def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    ...

main()
```

<details>
<summary><strong>Answer</strong></summary>

The last line runs `main()` on every import, not only when the file is run as a program: in the notebook, argparse reads the notebook's own command-line words, fails, and stops with exit code 2. Guard it: `if __name__ == "__main__": raise SystemExit(main())`. `__name__` is `"__main__"` only for `python digits.py ...`; on `import digits` it is `"digits"` (Theory S13).

</details>

**4. What is wrong here? A script runs `python digits.py predict models/nope.json x.txt > result.txt` every night, then reads the exit code. Name two problems.**

```python
try:
    output = commands[args.command](args)
except FileNotFoundError as err:
    print(f"error: {err}")
    return 0
```

<details>
<summary><strong>Answer</strong></summary>

(1) The error goes to **stdout**, so `> result.txt` saves it as if it were a result: `print(f"error: {err}", file=sys.stderr)`. (2) `return 0` tells the script that everything worked: an error the user can fix returns **1** (and wrong words get 2 from argparse).

</details>

## After the CORE: two activities

Two short activities for when your CORE is done (or at home): no new setup, a few minutes each.

### AI wrote this main() · ▶️ JUST RUN (about 5 minutes)

Someone asked an AI assistant for *"cmd_predict and main() for my digits.py, with robust and user-friendly
error handling"*. The answer is in `labkit/ai_samples/lab4_main_ai.py` (written by the instructor in the
style of an AI assistant's answer: not real AI output). It runs, and it looks careful. **It has three
problems.** Run the three cells, open the file in VS Code and read it, then write each problem down: what,
where, and why it matters to the night shift or to a test.

In [ ]:
# ▶️ JUST RUN · the AI's tool on a broken file from the inbox (line 12 lost a number in transfer)
from labkit.ai_samples import lab4_main_ai as ai
from labkit.lab4_tools import demo_model
ai_model = demo_model()            # a model for this activity, made by the finished tool
run_cli(ai.main, f"predict {ai_model} inbox/retiro_0005.txt", prog="digits_ai.py");

In [ ]:
# ▶️ JUST RUN · the AI's tool when the model file isn't there
run_cli(ai.main, "predict models/nope.json samples/parcel_021.txt", prog="digits_ai.py");

In [ ]:
# ▶️ JUST RUN · a test calls the AI's main([...]) and wants the exit code back
try:
    code = ai.main(["predict", ai_model, "samples/photo.png"])
    print("main returned", code)
except SystemExit as stop:
    print(f"main never returned: SystemExit({stop.code}). A test (or this notebook) that calls main([...]) "
          "stops here, and its other checks never run.")

**Your notes:**

1. Problem 1 (what, where, why):
2. Problem 2:
3. Problem 3:

<details>
<summary><strong>What we found (open after you've tried)</strong></summary>

1. **`sys.exit(1)` inside `cmd_predict`** (in the bare `except:`). It stops the whole program, so `main([...])`
   never returns: a test, or this notebook, can't read the exit code (cell 3). Commands should raise; only
   `main` decides the exit code, by returning it.
2. **Errors printed on stdout** (`print(f"Error: {error}")` in `main`, and the message in `cmd_predict`). In
   cell 2, the error line has no `stderr |` mark. A script that saves stdout as the result would save the
   error message as if it were a prediction.
3. **A bare `except:`** in `cmd_predict`. It catches everything, including Ctrl+C (`KeyboardInterrupt`) and
   real bugs, and replaces the real reason with "could not read the image file": in cell 1 the night shift
   never learns that line 12 has 27 numbers. Catch the errors you expect (`ValueError`, `FileNotFoundError`)
   and let the message through.

</details>

### Hand it to your neighbour · ▶️ JUST RUN (optional pairing, about 3 minutes)

`--help` is the only documentation the night shift has. Swap laptops with a neighbour (or, alone, pretend you
have never seen the code): using **only** the tool's own help, `python digits.py --help`, `train --help` and
`predict --help` (in a terminal, or with `run_cli` in a new cell), train a model and read one parcel. Then
write down, here:

- **One thing that was clear:**
- **One thing that confused me** (a word in the help, an error that didn't say what to do, an option I had to
  guess):
- **One change I'll make** (a `help=` text, an error message):

## OPTIONAL: extra practice

Extra practice for when your CORE is done. Pick any task, in any order: each one is independent, has a `check()` and hints, and nothing later depends on it.

| Task | What you'll learn | Difficulty | Typical time |
|---|---|---|---|
| **ENG1** Bullet-proof and tested | reject a broken model file before you trust it, and write tests that catch bugs (mutation testing) | moderate | 15–20 min |
| **SCI1** Measure it from the command line | a new subcommand, evaluate: accuracy per digit, timing, and a JSON report | gentle | 10–15 min |
| **BLD1** The overnight batch | a folder in, a CSV out: glob, a timestamped backup, archiving with shutil | moderate | 12–15 min |

In [ ]:
# ▶️ JUST RUN · tools for the OPTIONAL and ADVANCED tasks (run this once)
import json
from labkit.lab4_tools import copy_inbox, show_inbox_result, hygiene_report, score_strips, show_strip
from labkit.lab4_tools import load_web_app, serve_app_in_background, serve_in_background, ask
from labkit.reference import lab4 as finished      # the finished tool: provided cells use its model reader
from labkit.lab4_tools import ready, have_model     # False (with a line saying why) while something is missing

### OPTIONAL · ENG1 · Bullet-proof and tested · Engineer track · ✏️ WRITE CODE

A model file can be valid JSON and still be broken: an old version, a missing digit, a centroid cut short. Reject it at the door, then prove it with tests.

**What you'll learn:** reject a broken model file with a clear ValueError, write tests for digits.py, and measure how good they are with mutation testing · **Difficulty:** moderate · **Typical time:** 15–20 min

**Signature**

```python
def check_model(model: dict) -> None      # in digits.py; load_model already calls it
```

**Toy example**

```python
check_model(good_model)                 # None: nothing to complain about
check_model({**good_model, "version": 2})   # ValueError: model version 2 is not supported (this digits.py reads version 1)
check_model(model_without_centroids)    # ValueError: the model has no 'centroids'
```

**Part 1 · `check_model`** (in `digits.py`, under the ENG1 banner). Raise `ValueError` for: something that isn't a
dict; a missing key (`"format"`, `"version"`, `"n_train"`, `"image_size"`, `"centroids"`); a version other than
`MODEL_VERSION`; centroids that aren't exactly `"0"` to `"9"`; a centroid that isn't a list of
`image_size * image_size` numbers. The provided `load_model` already calls `check_model(model)` after its format
check, so once you write it, every model file is checked at the door. Why `ValueError` and not `KeyError`? `main()` catches `ValueError` and prints one line;
a `KeyError` would show the night shift a traceback.

**Part 2 · tests** (below the check): tests for `digits.py` that a buggy version can't pass.

In [ ]:
# ▶️ JUST RUN · check your work (this cell reads digits.py again, as you last saved it)
digits = reload_digits()
check("ENG1", digits.check_model);   # also checks a model file through load_model

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

One `if` per way the model can be broken, each raising ValueError with a message that says what is wrong. Check the type before you use a value: `isinstance(model, dict)` first, then the keys, then the values.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `if not isinstance(model, dict): raise ValueError(...)`; then `for key in ("format", "version", "n_train", "image_size", "centroids"):` raise if `key not in model`.
2. Raise if `model["version"] != MODEL_VERSION`, and if `sorted(model["centroids"]) != [str(d) for d in range(10)]`.
3. `size = model["image_size"]`; for each centroid: raise if it isn't a list of `size * size` values, or if a value isn't an int or a float (a bool doesn't count).

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

#### Part 2 · Tests that catch bugs

A test runs one command and checks what a user would check. `tests/test_example.py` (in the kit) has two;
open it in VS Code: each test calls `digits.main([...])` through a small `run()` helper that returns
`(exit code, stdout, stderr)`, and uses `tmp_path`, a new empty folder, for the files it writes.

1. Copy `tests/test_example.py` to `tests/test_digits.py`, and write **at least three tests** of your own:
   for example train with `--limit` then train again without `--force`; predict a `.csv` and a `.txt`
   image; a model file of another version.
2. Run them (the first cell below), until they all pass on your `digits.py`.
3. Then the real question: **are they any good?** `labkit.mutate` runs them on six buggy versions of
   `digits.py`. A test suite that passes on a buggy version didn't notice the bug.

**The AI moment:** ask an AI assistant to *"write pytest tests for the main() of this digits.py"* (paste
`digits.py`, never personal data), save its answer as `tests/test_ai.py`, and run `labkit.mutate` on it too.
Whose tests catch more mutants, yours or the AI's? Why?

In [ ]:
# ▶️ JUST RUN · run every test in tests/ (the same as typing it in a terminal, in the kit folder)
terminal("python -m labkit.minitest tests");

In [ ]:
# ▶️ JUST RUN · how many of the six buggy versions of digits.py do your tests catch? (10–20 s)
terminal("python -m labkit.mutate tests/test_digits.py");

**Think about it.** Which mutant survived longest? Which test caught it in the end, and what did that test
check that the others didn't (the exit code, stdout, stderr, or a file)? The same tests also run with pytest:
`uv run --with pytest pytest -q` (it needs internet the first time).

### OPTIONAL · SCI1 · Measure it from the command line · Scientist track · ✏️ WRITE CODE

Ops wants to know how good the model is before they trust it with real parcels: a new subcommand, evaluate, measures it on any labelled digit file.

**What you'll learn:** a new subcommand end to end: accuracy overall and per digit, timing, and an optional JSON report · **Difficulty:** gentle · **Typical time:** 10–15 min

**Signature**

```python
def cmd_evaluate(args: argparse.Namespace) -> str      # in digits.py
# evaluate DATA --model MODEL [--limit N] [--report FILE]
```

**Toy example**

```python
# python digits.py evaluate data/mnist_test_2k.csv.gz --model models/centroids.json --report output/report.json
Evaluated models/centroids.json on 2,000 images in S.S s
Accuracy: NN.N% (n of 2,000)
Per digit: 0: NN% · 1: NN% · 2: NN% · ... · 9: NN%
Report saved: output/report.json
```

Three places to edit in `digits.py`: the function `cmd_evaluate` (under the SCI1 banner); an `evaluate`
subparser in `build_parser` (`data`; `--model`, required; `--limit`, an int; `--report`, default `None`); and
`"evaluate": cmd_evaluate` in `main`'s dispatch dict. The report is a JSON file with the keys `model`, `data`,
`n`, `correct`, `accuracy` (a fraction from 0 to 1, 4 decimals), `per_digit` (`{"0": 0.95, ...}`) and `seconds`.

In [ ]:
# ▶️ JUST RUN · check your work (this cell reads digits.py again, as you last saved it)
digits = reload_digits()
check("SCI1", digits.cmd_evaluate);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Two dicts keyed by label: how many images of each digit you saw, and how many of those were read right. Accuracy is their totals divided; each digit's share is right / seen.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `centroids = centroids_of(load_model(Path(args.model)))`; `images, labels = load_digits(args.data, limit=args.limit)`; start a stopwatch.
2. For each image and label: `seen[label] = seen.get(label, 0) + 1`; if `predict_centroid(image, centroids) == label`, do the same in `right`.
3. `accuracy = sum(right.values()) / len(labels)`; `per_digit = {str(d): round(right.get(d, 0) / seen[d], 4) for d in sorted(seen)}`.
4. Build the three lines (`:.1%` and `:.0%` turn fractions into percentages); with `args.report`, also `json.dump` the report dict (`indent=2`) and add the fourth line.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · YOUR evaluate on all 2,000 test images, with a report (about 2 s)
from labkit.lab4_tools import ready_command, have_model
digits = reload_digits()
if ready_command(digits, "evaluate") and have_model():
    run_cli(digits.main, "evaluate data/mnist_test_2k.csv.gz --model models/centroids.json --report output/report.json");
    if Path("output/report.json").is_file():
        print(Path("output/report.json").read_text(encoding="utf-8")[:300])

**Think about it.** Which digit does the average-digit model read worst? Does the number agree with what you
saw in `samples/`? Ops asks for "at least 80% on every digit": what would you answer, and which command shows
it?

### OPTIONAL · BLD1 · The overnight batch · Builder track · ✏️ WRITE CODE

Every night, the scanners drop image files in an inbox folder. One command must read them all, write one CSV, keep yesterday's CSV, and clear the inbox for tomorrow.

**What you'll learn:** a folder in, a CSV out: Path.glob, a timestamped backup, archiving with shutil, and one summary line (Theory S7) · **Difficulty:** moderate · **Typical time:** 12–15 min

**Signature**

```python
def cmd_batch(args: argparse.Namespace) -> str      # in digits.py
# batch INBOX --model MODEL [--out predictions.csv]
```

**Toy example**

```python
# python digits.py batch inbox --model models/centroids.json --out predictions.csv
Read 30 files: N predicted, M unreadable · wrote predictions.csv (backup: predictions_20261118_031500.csv) · moved them to inbox/archive/20261118_031500
# predictions.csv            file,prediction
#                            chamartin_0001.csv,7
#                            retiro_0005.txt,"error: retiro_0005.txt, line 12: 27 numbers, expected 28 like line 1"
```

The rules: the `.csv` and `.txt` files of INBOX, sorted by name (`Path.glob`); one row per file; a file it can't
read gets `error: ...` in its row and the run goes on (one bad file must not stop the night); an existing CSV is
first copied to `<name>_YYYYmmdd_HHMMSS.csv` (`shutil.copy2`); the files it read move to
`INBOX/archive/YYYYmmdd_HHMMSS/` (`shutil.move`); one summary line is returned. Also add a `batch` subparser in
`build_parser` and `"batch": cmd_batch` in `main`'s dict. **Why the backup and the archive?** At 3 a.m. nobody is
there to undo a mistake: nothing is ever overwritten or lost, and each night's files stay together.

In [ ]:
# ▶️ JUST RUN · check your work (this cell reads digits.py again, as you last saved it)
digits = reload_digits()
check("BLD1", digits.cmd_batch);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

One timestamp for the whole run: `stamp = datetime.now().strftime("%Y%m%d_%H%M%S")`. Read every file first (inside try/except ValueError), then write the CSV, then move the files.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `inbox = Path(args.inbox)`; `centroids = centroids_of(load_model(Path(args.model)))`; `files = sorted(list(inbox.glob("*.csv")) + list(inbox.glob("*.txt")))`.
2. For each file: try to read it (by suffix) and append `{"file": file.name, "prediction": ...}`; `except ValueError as err:` append `"error: {err}"` instead.
3. `out = Path(args.out)`; if it exists, `shutil.copy2(out, out.with_name(f"{out.stem}_{stamp}{out.suffix}"))`; write the rows with `csv.DictWriter` (`newline=""`).
4. `archive = inbox / "archive" / stamp`; `archive.mkdir(parents=True, exist_ok=True)`; `shutil.move` every file into it; return one line.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · YOUR batch on a copy of the kit's inbox (30 files), then what it left behind
from labkit.lab4_tools import ready_command, have_model, copy_inbox, show_inbox_result
digits = reload_digits()
if ready_command(digits, "batch") and have_model():
    tonight = copy_inbox()
    run_cli(digits.main, f"batch {tonight.as_posix()} --model models/centroids.json --out output/predictions.csv");
    show_inbox_result(tonight, "output/predictions.csv")

**Run the cell again** (it copies the inbox again, as if a new night started): where did yesterday's CSV go?
**Think about it:** why is the timestamp `YYYYmmdd_HHMMSS` and not `18-11-2026 3.15`? (Sort the folder names
alphabetically, and remember which characters a file name can't hold on Windows.)

## ADVANCED: for students who move faster

For students who move faster. These tasks are more open: a sanity check plus a cell where you explain your result. Nothing here is needed for the next lab.

| Task | What you'll learn | Difficulty | Typical time |
|---|---|---|---|
| **ENG2** Hygiene | logging to stderr with --verbose, .gitignore, git, and a README with usage and the AI declaration | demanding | 20–25 min |
| **SCI2** A model registry | a second kind of model (a perceptron), chosen by the file's format: dispatch with a dict | demanding | 20–25 min |
| **BLD2** A web API for the canvas | FastAPI (Theory S16): your model behind the drawing page, a Pydantic model as the contract, 127.0.0.1 only | demanding | 20–25 min |
| **Challenge** Ship more | read whole postcode strips, turn the tool into a package, or schedule the nightly batch | open-ended | open-ended |

In [ ]:
# ▶️ JUST RUN · tools for the ADVANCED tasks (the same as at the start of OPTIONAL: run this once)
import json
from labkit.lab4_tools import copy_inbox, show_inbox_result, hygiene_report, score_strips, show_strip
from labkit.lab4_tools import load_web_app, serve_app_in_background, serve_in_background, ask
from labkit.reference import lab4 as finished      # the finished tool: provided cells use its model reader
from labkit.lab4_tools import ready, have_model     # False (with a line saying why) while something is missing

### ADVANCED · ENG2 · Hygiene · Engineer track · ✏️ WRITE CODE

A tool other people run needs a few habits: it can explain what it is doing (--verbose), it keeps data and models out of version control, and it has a README.

**What you'll learn:** logging to stderr with a level chosen by --verbose; a .gitignore; git init and three small commits; a README with usage and the AI declaration · **Difficulty:** demanding · **Typical time:** 20–25 min

**Signature**

```python
def setup_logging(verbose: bool) -> logging.Logger
```

**Toy example**

```python
log = setup_logging(verbose=True)
log.info("training on %d images", 10000)     # stderr: 15:20:05 INFO training on 10000 images
log = setup_logging(verbose=False)
log.info("...")                               # nothing (INFO is below WARNING)
log.warning("model is 30 days old")           # stderr: 15:20:05 WARNING model is 30 days old
```

**Step 1 · logging** (in the cell below first; then in `digits.py`): `setup_logging` returns the logger
`logging.getLogger("digits")` with one handler that writes to **stderr**, at level INFO with `verbose=True` and
WARNING otherwise. In `digits.py`: `parser.add_argument("-v", "--verbose", action="store_true")` on the main
parser, `log = setup_logging(args.verbose)` in `main`, and a `log.info(...)` line in each command. Why stderr?
stdout is for results: `--verbose` must never change what a script reads.

**Step 2 · hygiene** (in a terminal, in the kit folder): add `data/` and `models/` to `.gitignore` (why: data
files are big, licensed, and sometimes personal; models can be rebuilt); `git init`; three small commits with
clear messages (e.g. "Add build_parser with train and predict"); fill in the **Usage** section of `README.md`
(it ends with the AI-use declaration). The checklist cell below reads your kit folder and says what is done.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

import logging
import sys


def setup_logging(verbose: bool) -> logging.Logger:
    """Return the "digits" logger, writing to stderr: INFO and up with verbose=True, else WARNING and up."""
    # TODO: logger = logging.getLogger("digits"); remove its old handlers (logger.handlers.clear())
    #       add a logging.StreamHandler(sys.stderr) with a Formatter("%(asctime)s %(levelname)s %(message)s")
    #       set the level (logging.INFO or logging.WARNING); logger.propagate = False; return logger
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("ENG2", setup_logging);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

A logger, a handler (where the lines go: sys.stderr) and a level (which lines go there). Clear the old handlers first, or running the cell twice prints every line twice.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `logger = logging.getLogger("digits")`; `logger.handlers.clear()`.
2. `handler = logging.StreamHandler(sys.stderr)`; `handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%H:%M:%S"))`; `logger.addHandler(handler)`.
3. `logger.setLevel(logging.INFO if verbose else logging.WARNING)`; `logger.propagate = False`; `return logger`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · see it: the same two log lines, with and without verbose (they go to stderr)
import sys
if ready(setup_logging):
    for verbose in (True, False):
        print(f"verbose={verbose}:", flush=True)
        log = setup_logging(verbose)
        log.info("training on %d images", 10000)
        log.warning("the model file is 30 days old")
        sys.stderr.flush()

In [ ]:
# ▶️ JUST RUN · the hygiene checklist: it reads your kit folder (it changes nothing)
hygiene_report();

### Explain your result (ENG2)

- Which of your three commit messages would help someone in six months? Why?
- Run `python digits.py --verbose predict ... > result.txt` in a terminal: where do the log lines go, and
  what is in `result.txt`?
- What else should never be committed? (Think of Lab 3's scanner export.)

### ADVANCED · SCI2 · A model registry · Scientist track · ✏️ WRITE CODE

Ops may want a second kind of model one day. If each model file says what it is (its "format"), one function can read any of them: a registry.

**What you'll learn:** a second model (a multiclass perceptron) and a registry that picks the right predict function from the model's "format" · **Difficulty:** demanding · **Typical time:** 20–25 min

#### The idea, without code

The average-digit model compares an image with 10 averages. A **perceptron** learns 10 lists of weights
instead, one per digit: to read an image, multiply each pixel by the digit's weight for that pixel, add them
up (plus a bias), and the digit with the biggest total wins. It learns from its mistakes: when it reads a 7
as a 1, it adds the image to the 7's weights and subtracts it from the 1's. A few passes over the training
images (*epochs*) are enough.

Saved as a file it looks like `{"format": "garabato-perceptron", "version": 1, "image_size": 14,
"weights": {"0": [196 weights, then the bias], ...}}`. The **registry** is a dict from format to function:
`{"garabato-centroids": ..., "garabato-perceptron": predict_perceptron}`; `predict_with(model, image)` looks
up `model["format"]` and calls the right one (an unknown format: `ValueError`). In `digits.py` that means:
`train --kind perceptron`, and `cmd_predict` calling `predict_with`. It trains on 14 x 14 images
(`downsample`): four times fewer weights.

#### The implementation

```python
def train_perceptron(images: list[list[int]], labels: list[int], epochs: int = 3) -> dict
def predict_perceptron(model: dict, image: list[int]) -> int      # ties: the smallest label
def predict_with(model: dict, image: list[int]) -> int            # the registry
```

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

from operator import mul
from labkit.reference.lab1 import predict_centroid


def train_perceptron(images: list[list[int]], labels: list[int], epochs: int = 3) -> dict:
    """A multiclass perceptron: one weight per pixel plus a bias (the last weight), per label."""
    # TODO: weights = {label: [0] * (len(images[0]) + 1) for each label}; for each epoch, for each image:
    #       guess = the label with the biggest score; if wrong: add the image to weights[label] (bias + 1),
    #       subtract it from weights[guess] (bias - 1). Return the model dict of the card.
    pass


def predict_perceptron(model: dict, image: list[int]) -> int:
    """The label with the biggest score (sum of weight * pixel, plus the bias); ties: the smallest label."""
    pass


def predict_with(model: dict, image: list[int]) -> int:
    """Read `image` with any saved model: its "format" chooses the function. Unknown format: ValueError."""
    # TODO: a dict {format: function}; centroid models: predict_centroid(image, digits.centroids_of(model))
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("SCI2", predict_with);   # also checks that train_perceptron learns

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Score = sum(map(mul, weights, image)) + bias; the guess is the label with the biggest score. The registry is a dict from the format text to a function; look it up, then call it.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `train_perceptron`: `weights = {label: [0] * (size + 1) for label in sorted(set(labels))}`; for each epoch and each (image, label): compute every label's score, take the best (ties: smallest label); if it isn't `label`, add the image to `weights[label]` and subtract it from `weights[guess]` (and the bias: +1, -1).
2. Return `{"format": "garabato-perceptron", "version": 1, "image_size": round(size ** 0.5), "weights": {str(label): w ...}}`.
3. `predict_with`: `predictors = {"garabato-centroids": lambda m, x: predict_centroid(x, digits.centroids_of(m)), "garabato-perceptron": predict_perceptron}`; raise ValueError for an unknown `model.get("format")`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · the experiment: a perceptron on the 10,000 training images (14 x 14), 2 epochs (a few seconds),
# then both models on the 2,000 test images, through YOUR registry
import time
if ready(train_perceptron, predict_perceptron, predict_with) and have_model():
    train_14, train_labels = load_digits("data/mnist_train_10k_14x14.csv.gz")
    test_14, test_labels = load_digits("data/mnist_test_2k_14x14.csv.gz")
    test_28, _ = load_digits("data/mnist_test_2k.csv.gz")
    start = time.perf_counter()
    perceptron = train_perceptron(train_14, train_labels, epochs=2)
    print(f"Trained the perceptron in {time.perf_counter() - start:.1f} s")
    centroid_model = finished.load_model(Path("models/centroids.json"))
    for name, model, images in (("average digits (28 x 28)", centroid_model, test_28),
                                ("perceptron (14 x 14)", perceptron, test_14)):
        start = time.perf_counter()
        right = sum(1 for image, label in zip(images, test_labels) if predict_with(model, image) == label)
        size_kb = len(json.dumps(model)) / 1024
        print(f"{name:<26} {right / len(test_labels):6.1%}   {1000 * (time.perf_counter() - start) / len(images):5.2f} ms per digit"
              f"   {size_kb:5.0f} KB as JSON")

### Explain your result (SCI2)

- Which model reads more digits right, and which is faster per digit? Which would you ship, and why?
- Run it again with `epochs=3`. What changes? Is that difference bigger than the noise you would expect from
  2,000 test images?
- A third kind of model arrives next month. What changes in `predict_with`, and what stays the same?

### ADVANCED · BLD2 · A web API for the canvas · Builder track · ✏️ WRITE CODE

Third face of the same model: the notebook, the terminal, and now the browser. tools/draw_digit.html can ask a server for a prediction; you write that server's /predict with FastAPI.

**What you'll learn:** a small web API: a Pydantic model that says what /predict accepts, the route itself, and a server only this computer can reach (127.0.0.1) (Theory S16) · **Difficulty:** demanding · **Typical time:** 20–25 min

**Signature**

```python
class Drawing(BaseModel)                   # in web_app.py: what POST /predict accepts
def predict(drawing: Drawing) -> dict      # in web_app.py: the route POST /predict
```

**Toy example**

```python
POST /predict  {"pixels": [0, 0, ... 784 numbers]}   ->  200  {"top3": [[7, 1.0], [1, 0.85], [9, 0.71]]}
POST /predict  {"pixels": [0, 0, ... 783 numbers]}   ->  422  pixels: List should have at least 784 items ...
POST /predict  pixels=1,2,3                          ->  422  JSON decode error
```

**Why a web API?** The page in `tools/draw_digit.html` runs in the browser, which can't run Python. It sends
the drawing as JSON, `{"pixels": [784 numbers]}`, to `POST /predict`, and shows the answer. A program that
answers such requests is a web API.

**Why FastAPI and a Pydantic model?** Anything can arrive in a request: 783 pixels, a pixel of 300, text that
isn't JSON. Theory S16 describes a request body with a class (if you haven't reached S16 yet, this card is your
first look); FastAPI then refuses every body that breaks the description with status **422**, and an
explanation, *before* your function runs. So `predict` itself is one line: it only ever sees a valid drawing.

| You write (`web_app.py`, under the banner) | Provided in `web_app.py` |
|---|---|
| `class Drawing(BaseModel)`: one field, `pixels`, a list of exactly 784 `Pixel` | `app = FastAPI(...)`, `GET /` (the drawing page), `top3(pixels)` (the 3 closest average digits), `Pixel` (an `int` from 0 to 255) |
| `@app.post("/predict")` and the body of `predict` | `if __name__ == "__main__":` start the server with uvicorn on **127.0.0.1**, port 8000 |

**Before you start:** FastAPI and uvicorn are not part of the standard library: `uv sync --group web` in the kit
folder (it needs internet; then restart the kernel). No FastAPI on this computer? The by-hand server at the end of
this task runs the same page without it.

**Where:** `web_app.py`, next to `digits.py` (look for the banner `# ===== YOUR CODE HERE =====`). Write it there, **save
the file**, then run the check cell below.

In [ ]:
# ▶️ JUST RUN · check your work (it reads web_app.py again, as you last saved it; it needs FastAPI)
web = load_web_app()             # None, with one line saying why, if FastAPI isn't installed
if web is not None:
    check("BLD2", web.predict);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

A Pydantic field is written like a type hint inside the class, with `Field(...)` for its limits, as `amount: float = Field(gt=0)` in Theory S16. A route is a decorator on the line above the function.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. In `Drawing`, replace `pass` with one field: `pixels: list[Pixel] = Field(min_length=784, max_length=784)`.
2. On the line above `def predict`, add the decorator `@app.post("/predict")`.
3. In `predict`, replace `pass` with `return {"top3": top3(drawing.pixels)}`. Save `web_app.py`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    class Drawing(BaseModel):
        pixels: list[___] = Field(min_length=___, max_length=___)


    @app.___("/predict")
    def predict(drawing: Drawing) -> dict:
        return {"top3": top3(drawing.___)}

</details>

<details>
<summary><strong>Hint 4 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · see it: YOUR web_app on a real local server, four requests, then it stops (about 2 s)
if web is not None and ready(web.predict) and have_model():
    url, stop = serve_app_in_background(web.app)
    drawing = [int(v) for v in Path("samples/parcel_021.txt").read_text(encoding="utf-8").split()]
    ask(url + "predict", {"pixels": drawing}, "a drawing")
    ask(url + "predict", {"pixels": drawing[:783]}, "one pixel short")
    ask(url + "predict", {"pixels": [300] + drawing[1:]}, "a pixel of 300")
    ask(url + "predict", b"pixels=1,2,3", "not JSON")
    stop()

**Now in the browser** (VS Code or Jupyter on your laptop; on Colab the browser can't reach the server):

1. In a terminal, in the kit folder: `python web_app.py` (or `uv run python web_app.py`). It prints the
   address and keeps running: a server waits for requests.
2. Open **http://127.0.0.1:8000/**, draw a digit, press **Ask the server (Lab 4)**.
3. Open **http://127.0.0.1:8000/docs**: FastAPI wrote this page from your code. Open `POST /predict`, press
   *Try it out*, send `{"pixels": [1, 2, 3]}` and read the 422.
4. Back in the terminal, **Ctrl+C** stops the server.

#### No FastAPI on this computer? The by-hand server

Python's own `http.server` can serve the same page. Without Pydantic, every check of your `Drawing` has to be
written by hand: open `labkit/lab4_tools.py`, find `handle_by_hand`, and compare its 15 lines with your one
field. Run the first cell, open the address it prints, draw, ask; then run the second cell to stop it.

In [ ]:
# ▶️ JUST RUN (optional) · the by-hand server (Python's http.server, no FastAPI): open the address it prints
if have_model():
    by_hand_url, stop_by_hand = serve_in_background()

In [ ]:
# ▶️ JUST RUN (optional) · stop the by-hand server
if "stop_by_hand" in globals():
    stop_by_hand()

### Explain your result (BLD2)

- Your `predict` is one line. Who checked that the drawing has 784 pixels from 0 to 255, and when? What did
  the request with one pixel short get back, and from whom?
- Compare your `Drawing` with `handle_by_hand` in `labkit/lab4_tools.py`. What does the by-hand version
  check that yours doesn't?
- Why `host="127.0.0.1"` and not `0.0.0.0`? Who could reach the server in each case, on the campus Wi-Fi?

### ADVANCED · Challenge: Ship more · ✏️ WRITE CODE

**What you'll learn:** taking a tool further: a real postcode reader, a package, or scheduling · **Difficulty:** open-ended · **Typical time:** open-ended (you can continue at home)

Choose one. AI is your pair programmer here: verify what it suggests, and say how you used it in your Engineering log.

**Ideas to try**

- **(a) Read a whole strip.** `strips/` holds 10 postcode strips, 28 x 140 pixels each (5 digits side by side), and `strips/strips_truth.csv`. Write `read_strip(path, centroids) -> str` (cells below): first split by fixed columns (5 slots of 28); then handle the strips whose digits are irregularly spaced, by finding the blank columns between digits. Score it with `score_strips`; check each answer against `data/valid_postcodes.txt`. Then add it to `digits.py` as `read-strip STRIP.txt --model MODEL`.
- **(b) A package.** `digits.py` keeps growing (strips, logging, a web API). Split it into a package `garabato_digits/` (Theory S13): `__init__.py`, one module per job (for example `models.py` for the model file: new, save, load, check; `cli.py` for `build_parser`, the commands and `main`) and `__main__.py` (`from .cli import main` and `raise SystemExit(main())`), so `python -m garabato_digits --help` works from the kit folder. Which functions belong together, and which module imports which?
- **(c) The nightly run.** Write (don't run) the line that starts the batch (BLD1) every night at 03:00, with its output appended to `logs/batch.log`: a crontab line (macOS, Linux) and a `schtasks` line (Windows). What happens to the exit code? Who is told when it isn't 0?

**Rules**

- Standard library only for (a); explain every line you keep.
- Never paste personal data into an AI tool.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

from labkit.data import read_image_grid
from labkit.reference.lab1 import predict_centroid


def read_strip(path, centroids: dict[int, list[float]]) -> str:
    """Read the 5 digits of a 28-row postcode strip and return them as text, e.g. "28013"."""
    # TODO: pixels = read_image_grid(path); width = len(pixels) // 28; first: 5 fixed slots of 28 columns,
    #       each pasted into a 28 x 28 image and read with predict_centroid(image, centroids)
    pass

In [ ]:
# ▶️ JUST RUN · look at a strip (a regular one, then an irregular one)
show_strip("strips/strip_02.txt")
show_strip("strips/strip_09.txt")

In [ ]:
# ▶️ JUST RUN · score YOUR read_strip on the 10 strips (the model from the train step)
strip_score = score_strips(read_strip, "models/centroids.json")

### Engineering log

Short notes are fine. Edit this cell as you go.

- **What I tried:**
- **What I measured:**
- **What I'd do next:**

## AI-use declaration (about 1 minute)

This course asks you to use AI critically and to say how you used it. This cell stays in your notebook:
nobody collects or grades it. Please don't write your name.

- **Tools I used** (or "none"):
- **For which tasks** (e.g. C2, ENG1):
- **One prompt I used** (copy it, or summarise it in one line):
- **How I checked the answer** (check(), my own test, reading the code, a classmate…):
- **Something the AI got wrong, or that I changed:**
- **One line of AI-written code I can now explain in my own words:**

## Before you go (private, 30 seconds)
- One thing I can do now that I couldn't at the start of this lab:
- One thing I'm still unsure about:

## What's next

- **Catch-up pack**, on IE Campus after class: `lab4_solution.ipynb` (CORE and OPTIONAL solutions), `lab4_express.ipynb` (the minimum path in about 25 minutes, with the same checks) and `lab4_what_you_missed` (a one-page summary).
- **ADVANCED solutions** (including the Challenge) come with the Lab 5 kit, in `solutions_previous/`, so the Challenge stays open until then.
- **Next:** Lab 5 · The Model Arena, Tue 1 Dec, 12:30–13:50, MMB-703. The kit appears on IE Campus 48 hours before.
- **The League** is opt-in and uses pseudonyms: the rules are in `LEAGUE.md`.
- **The same declaration is in `README.md`**, next to `digits.py`: a tool travels without its notebook.
- **Missed something?** `labkit/reference/lab4.py` is a finished `digits.py` (CORE and OPTIONAL), with comments; `lab1.py` to `lab3.py` next to it cover the earlier labs.
- **Lab 3's ADVANCED solutions**, the Challenge included, are in `solutions_previous/`.
- **Before Lab 5:** the catch-up notebook `bridge_4to5` (15 minutes: turn the average-digit model into a class) and a track card; if you can, run `uv sync --group nn` at home (it installs scikit-learn for Lab 5).